In [ ]:
#Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Importar las librerias
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from xgboost import XGBRegressor
import joblib
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

In [ ]:
#Proyeccion de una gestion 2026
ruta_20253 = '/content/drive/My Drive/Anexos-CD/Predicciones/XGBoost/proyeccion_3T_2025_RandomForest.csv'
df3t = pd.read_csv(ruta_20253)
df_base_2026 = df3t.copy()

df_base_2026 = df_base_2026.drop_duplicates(
    subset=["matricula", "curso", "nivel", "materia", "sexo"]
)

df_base_2026 = df_base_2026.drop(columns=["nota", "trimestre"])

# Quitamos la nota (será predicha)
df_2026 = []

map_trimestre = {
    "1T": 1,
    "2T": 2,
    "3T": 3
}

for t, t_num in map_trimestre.items():
    temp = df_base_2026.copy()
    temp["trimestre"] = t
    temp["trimestre_num"] = t_num
    df_2026.append(temp)

df_2026 = pd.concat(df_2026, ignore_index=True)


encoder = joblib.load("/content/drive/My Drive/Anexos-CD/Encoders/encoder_rf.joblib")
rf = joblib.load("/content/drive/My Drive/Anexos-CD/Modelos/modelo_random_forest_notas.joblib")

cat_cols = ["curso", "materia", "nivel", "sexo"]

X26_cat = encoder.transform(df_2026[cat_cols])
X26_num = df_2026[["trimestre_num", "matricula"]]

X26 = np.hstack([X26_num, X26_cat])

df_2026["nota"] = rf.predict(X26)
df_2026["nota"] = df_2026["nota"].round().clip(1, 100).astype(int)
print(
    df_2026.duplicated(
        subset=["matricula", "curso", "materia", "trimestre"]
    ).sum()
)
df_2026.to_csv(
    "/content/drive/My Drive/Anexos-CD/Predicciones/XGBoost/proyeccion_gestion_2026_randomforest.csv",
    index=False
)
df_base_2026.info()